<h1 style="text-align:center">00-Welcome to the XiangShan Tutorial！</h1>




In this section, we will give some notes on this tutorial.

- Cells that start with %%bash are Bash scripts; the rest are Python code.
- Lines that start with # are comments.


You can click ▶ in the top-left corner of a cell to run that single cell; the output will be displayed below the cell.

In [ ]:
%%bash
echo "Welcome to the XiangShan Tutorial!"

Each cell has its own working directory and environment variables, so some commands may need to be rerun. If you execute these commands directly in the shell, you can skip the repetitive parts, e.g. `source env.sh`.

In [ ]:
%%bash
# Change the working directory in this cell.
cd ../
pwd

In [ ]:
%%bash
# Changing the working directory in other cells does not affect this cell.
pwd

XiangShan has design documentation synchronized with development; the GitHub repository is [https://github.com/OpenXiangShan/XiangShan-Design-Doc](https://github.com/OpenXiangShan/XiangShan-Design-Doc)

We have also deployed the design documentation website at [https://docs.xiangshan.cc/projects/design](https://docs.xiangshan.cc/projects/design)

The remaining steps will be completed in the xs-env repository. This is a development environment suite; you can first look at the XiangShan folder in this repository, which contains the source code of the XiangShan processor.

<h1 style="text-align:center">First Run</h1>

In this section, we present the most basic workflow for building and running XiangShan.

The xs-env repository contains the environment setup scripts necessary for compiling and running XiangShan and can be cloned directly from GitHub.

In [ ]:
%%bash
# For this tutorial, the local directories have been preconfigured; therefore, you do not need to execute these commands.
# The following commands are provided for reference.

# git clone https://github.com/OpenXiangShan/xs-env.sh  # clone repository
# cd xs-env                                             # enter xs-env
# git checkout -b tutorial-2025 origin/tutorial-2025    # change branch
# sudo -s ./setup-tools.sh                              # Install the required dependencies. Feel free to switch to a different package manager.
# source setup.sh                                       # Tool Setup and Environment Verification
# ./update-submodule.sh                                 # Update submodule repositories

Then, we can getting start!

The build and execution of XiangShan rely on specific environment variables, which are provisioned by the `env.sh` script in xs-env.

This script must be sourced whenever a new terminal session is started; to automate this, you can add it to your `.bashrc`.

As shown in Section 00-welcome, within this tutorial each cell constitutes a fresh Bash environment; therefore, the script must be re-sourced in every cell.

In [ ]:
%%bash
cd ../ && source env.sh

- Running the code block above completes the environment variable setup.
- After the setup, go to `$NOOP_HOME` (`xs-env/XiangShan`) to build XiangShan.
- The build parameters will be introduced later.

We can also use the tree command to view the project structure.

In [ ]:
%%bash
cd ../ && tree -d -L 1

XiangShan provides hundreds of user-configurable parameters, including:

- `src/main/scala/top/Configs.scala` Defines the processor core parameters.
- SoC parameters are passed at build time via yaml files; `src/main/resources/config/Default.yaml` is the default configuration.

Press Ctrl+P to open file search, then type the file name above to jump to it quickly.

With the configuration finalized, we can proceed to build XiangShan!

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

# Warning⚠️：Building XiangShan is highly resource‑intensive; For this tutorial, we’ve prepared a prebuilt executable for you.
# Reference setup: 16 CPU cores, 64 GB RAM.
# make emu -j16 CONFIG=MinimalConfig

# Additional build options
# CONFIG=MinimalConfig  XiangShan configuration
# EMU_THREADS=4         Simulation thread count
# EMU_TRACE=1           Enable waveforms
# WITH_DRAMSIM=1        Simulate DRAM with DRAMSim3
# WITH_CHISELDB = 1     Enable ChiselDB
# WITH_CONSTANTIN = 1   Enable Constantin

The commands above will generate outputs like `build/emu` and `build/rtl`.

- build/rtl/*.sv is Verilog files generated by Chisel.
- build/emu is a simulation executable further compiled with Verilator.

You can run `./build/emu` to simulate XiangShan. 

Since we haven’t built emu in this tutorial, we’ll use the precompiled emu located in `${READY2RUN_HOME}`.

We’ll introduce the run-time arguments later.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

${READY2RUN_HOME}/emu \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --no-diff \
    2>/dev/null

# Some key runtime parameters.
# -i                        Workload path
# -C / -I                   Maximum cycle count / Maximum instruction count
# --diff=PATH / --no-diff   Reference model path / Disable difftest

Note: XiangShan prints performance counters to stderr at the end of a run, and the output is very large. We recommend always redirecting stderr to a file. In the example above, since we don’t need the counters, we redirect it to `/dev/null`; adjust as needed.

<h1 style="text-align:center">Build the workload using Nexus-AM</h1>

Nexus-AM is a bare-metal runtime and test-generation environment. It is lightweight and easy to use, implements basic system call interfaces and exception handlers, and supports multiple ISAs and configurations. 

The `am/` directory contains the Nexus-AM framework sources; `apps/` and `tests/` hold common workload sources, and you can create your own apps and tests.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $AM_HOME

tree -d -L 1

echo apps: $(ls ./apps)
echo tests: $(ls ./tests)

We start with the Hello, XiangShan sample (`apps/hello`). Replace "Hello, XiangShan" with "Welcome to XiangShan Tutorial" 

Then compile with `ARCH=riscv64-xs`.

The default riscv64 toolchain is `riscv64-unknown-elf-`, but here we use the GNU toolchain (`riscv64-linux-gnu-`), so set `LINUX_GNU_TOOLCHAIN=1`. 

Reference files:

- `am/arch/isa/riscv64.mk`
- `am/arch/riscv64-xs.mk`

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $AM_HOME/apps/hello

# Use sed to replace "Hello, XiangShan" with "Welcome to XiangShan Tutorial".
sed -i 's/Hello, XiangShan/Welcome to XiangShan Turtorial/' hello.c

# compiling
make ARCH=riscv64-xs LINUX_GNU_TOOLCHAIN=1

# check output
ls -l build

After compilation, the following three files will be generated:
- hello-riscv64-xs.bin：Program binary image (The ELF header and other metadata was removed) for emu.
- hello-riscv64-xs.elf：The program's ELF file.
- hello-riscv64-xs.txt：The program’s disassembly for debugging

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

# Use emu to run workload.
${READY2RUN_HOME}/emu -i $AM_HOME/apps/hello/build/hello-riscv64-xs.bin --no-diff 2>/dev/null

If the ARCH you pass isn’t supported, `make` will print all supported ARCH values. The `|| true` is only to swallow the non‑zero exit code so the notebook doesn’t error out. In normal use, just run `make ARCH=`.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $AM_HOME

make ARCH= || true


<h1 style="text-align:center">Run the RTL simulation</h1>

XiangShan's emu supports many options; run emu --help to see usage.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

${READY2RUN_HOME}/emu --help

Unlike the 01-first-run chapter, this chapter we'll run a more complex program: CoreMark (2 iterations). The binary is already prepared in the `Xiangshan/ready-to-run` folder.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

${READY2RUN_HOME}/emu \
    -i ./ready-to-run/copy_and_run.bin \
    --no-diff \
    2>/dev/null

We've also prepared a fault-injection simulation program for XiangShan; feel free to try running them.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

# 错误 1
${READY2RUN_HOME}/emu-alu-err \
    -i ./ready-to-run/coremark-2-iteration.bin \
    --no-diff \
    -C 10000 \
    2>/dev/null || true

To prevent overuse of server resources, we stop the simulation after 10,000 cycles.

<h1 style="text-align:center">NEMU: ISA Reference</h1>

We developed NEMU, a Spike-like ISA simulator. With targeted optimizations, NEMU achieves QEMU-class performance and exposes APIs to compare and verify XiangShan's architectural state.

NEMU provides two default configurations:
- xxx_defconfig：xxx Default settings for standalone run mode
- xxx-ref_defconfig：xxx As the default configuration for DiffTest co-simulation mode

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null

cd $NEMU_HOME
### If you are building NEMU for the first time,
### you will need to run the following command to configure it.
### Here we have configured it in advance.
# make menuconfig
make clean
make riscv64-xs_defconfig
make -j

make clean-softfloat
make riscv64-xs-ref_defconfig
make -j

Execute CoreMark on NEMU.

In [ ]:
%%bash

cd ../ && source env.sh >/dev/null

cd $NEMU_HOME

# 以批模式运行 workload
./build/riscv64-nemu-interpreter \
    -b \
    ${READY2RUN_HOME}/hello-riscv64-xs.bin

<h1 style="text-align:center">Difftest：ISA Co-simulation framework</h1>

We introduce DiffTest, an ISA co-simulation framework. Flow: whenever the RTL core (DUT) commits an instruction or updates state, the ISA simulator(REF) executes the same instruction; DiffTest compares architectural state between the DUT and the REF. On any mismatch it halts and reports an error; otherwise it continues. 

Run the workload on XiangShan and use NEMU for differential testing.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null

cd $NOOP_HOME

${READY2RUN_HOME}/emu \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --diff $NEMU_HOME/build/riscv64-nemu-interpreter-so \
    2>/dev/null

You can run workloads on the prebuilt XiangShan processor with injected bugs and use NEMU for differential testing.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null

cd $NOOP_HOME

${READY2RUN_HOME}/emu-alu-err \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --diff $NEMU_HOME/build/riscv64-nemu-interpreter-so \
    2>/dev/null || true # tutorial：添加 "|| true" 避免 notebook 报错，实际使用不需要

可以看到执行 pc = 0x80000dce 处的指令时，仿真和模拟器执行产生了不一样的结果，仿真的体系结构寄存器 a6 值为 0，而模拟器为 0x800040b6。

After DiffTest reports an error, rerun the simulation and enable waveform dumping around the reported failing cycle.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME
rm -f ./build/*.vcd

${READY2RUN_HOME}/emu-alu-err \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --diff $NEMU_HOME/build/riscv64-nemu-interpreter-so \
    -b 8000 \
    -e 10000 \
    --dump-wave \
    2>/dev/null || true

echo -n "Dump wave: "
realpath ./build/*.vcd

<h1 style="text-align:center">LightSSS：Lightweight Simulation Snapshot</h1>

Dumping full waveforms is slow and wastes disk space; failures usually occur near the DiffTest error point (where architectural state mismatch).

LightSSS runs without waveforms, restores to a short simulation before the failure, re-simulates, and records waveforms.

Mechanism: it uses the `fork()` system call to snapshot the process; the forked child sleeps and, when a bug is detected, wakes to rerun with waveform dumping enabled.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd $NOOP_HOME

rm -f ./build/*.vcd

${READY2RUN_HOME}/emu-alu-err \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --diff $NEMU_HOME/build/riscv64-nemu-interpreter-so \
    --enable-fork \
    2> /dev/null || true

echo -n "Dump wave: "
realpath ./build/*.vcd

If you see "the oldest checkpoint start to dump wave and dump nemu log...", LightSSS is active. The simulation will then restart from the latest snapshot and record waveforms.

<h1 style="text-align:center">ChiselDB：Debug-friendly structured database</h1>

ChiselDB is a structured database for aiding functional and performance debugging. It uses a DPI-C interface to log Chisel Bundle data into an SQLite database.

We provide a prebuilt simulator `emu-cdb-err` with an injected bug that forces all data released from L2 Cache to L3 Cache to a constant value.

Enable ChiselDB with `--dump-db` and turn on DiffTest; after running, DiffTest reports an error and a `.db` file is generated under `./build`.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd ${NOOP_HOME}

rm -f ./build/*.db

${READY2RUN_HOME}/emu-cdb-err \
    -i $NOOP_HOME/ready-to-run/linux.bin \
    --diff $NOOP_HOME/ready-to-run/riscv64-nemu-interpreter-so \
    --dump-db \
    2>linux.err || true

echo -n "Dump DB: "
realpath ./build/*.db

Then use SQLite to read the `.db` for analysis: query all TileLink transactions at address `0x800419c0`, and format the output with `./scripts/cache/convert_tllog.sh`.

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd ${NOOP_HOME}

DB=$(ls -t ./build/*db | head -n 1)
sqlite3 $DB "select * from TLLog where ADDRESS=0x800419c0" | sh ./scripts/cache/convert_tllog.sh

<h1 style="text-align:center">Checkpoint 与性能评估</h1>

使用 checkpoint 进行性能评估在学术界和工业界都有广泛应用。为了生成 checkpoint，我们首先需要准备 NEMU 环境，并得到 GCPT restorer

本节中会用到一些与 `xs-env/env.sh` 不同的路径和常量，为了方便使用，我们创建了一个 `07-checkpoint-env.sh`，在本节中我们将使用该脚本设置环境变量，您可以运行下面的单元格查看这些环境变量。在后续的单元格中我们仍将输出重定向到 `/dev/null` 减少干扰。

In [ ]:
%%bash
source 07-checkpoint-env.sh

In [ ]:
%%bash
source 07-checkpoint-env.sh >/dev/null

cd ${NEMU_HOME}
git submodule update --init

# 编译 simpoint
cd ${NEMU_HOME}/resource/simpoint/simpoint_repo
make clean
make

# 编译 NEMU
cd ${NEMU_HOME}
make clean
make riscv64-xs-cpt_defconfig
make -j8

cd ${NEMU_HOME}/resource/gcpt_restore
rm -rf ${GCPT_PATH}
make -C ${NEMU_HOME}/resource/gcpt_restore/ O=${GCPT_PATH} GCPT_PAYLOAD_PATH=${PAYLOAD_PATH}/${WORKLOAD}.bin CROSS_COMPILE=riscv64-linux-gnu-

接下来，我们需要使用 NEMU 运行要进行切片的程序，来收集程序行为

In [ ]:
%%bash
source 07-checkpoint-env.sh >/dev/null

rm -rf $RESULT_PATH

_LOG_PATH=$LOG_PATH/profiling
mkdir -p $_LOG_PATH

# 使用 GCPT 作为镜像
# -w：要加载的实际负载为 WORKLOAD
# -C：使用 profiling 配置运行 NEMU
$NEMU ${GCPT} \
    -D ${RESULT_PATH} \
    -w ${WORKLOAD} \
    -C profiling \
    -b \
    --simpoint-profile \
    --cpt-interval ${CHECKPOINT_INTERVAL} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt)


进而，使用 simpoint 对采集到的程序行为进行聚类分析

In [ ]:
%%bash
source 07-checkpoint-env.sh >/dev/null

CLUSTER=${RESULT_PATH}/cluster/${WORKLOAD}
mkdir -p ${CLUSTER}

random1=`head -20 /dev/urandom | cksum | cut -c 1-6`
random2=`head -20 /dev/urandom | cksum | cut -c 1-6`

_LOG_PATH=$LOG_PATH/cluster
mkdir -p $_LOG_PATH

$SIMPOINT \
    -loadFVFile ${PROFILING_RESULT_PATH}/${WORKLOAD}/simpoint_bbv.gz \
    -saveSimpoints ${CLUSTER}/simpoints0 \
    -saveSimpointWeights ${CLUSTER}/weights0 \
    -inputVectorsGzipped \
    -maxK 3 \
    -numInitSeeds 2 \
    -iters 1000 \
    -seedkm ${random1} \
    -seedproj ${random2} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt)


最后，使用 NEMU 重新运行需要采样的程序片段，生成 checkpoint。

checkpoint 文件内包括需要执行的程序段，也包括 checkpoint 起始位置时的内存状态和处理器体系结构状态（通用寄存器堆，CSR）。

In [ ]:
%%bash
source 07-checkpoint-env.sh >/dev/null

CLUSTER=${RESULT_PATH}/cluster
_LOG_PATH=${LOG_PATH}/checkpoint
mkdir -p ${_LOG_PATH}

$NEMU ${GCPT} \
    -D ${RESULT_PATH} \
    -w ${WORKLOAD} \
    -C checkpoint \
    -b \
    -S ${CLUSTER} \
    --cpt-interval ${CHECKPOINT_INTERVAL} \
    > >(tee ${_LOG_PATH}/${WORKLOAD}-out.txt) 2> >(tee ${_LOG_PATH}/${WORKLOAD}-err.txt)


我们可以使用 emu 运行一下采集到的 checkpoint，看看效果。

emu 检测到文件是 gzip 压缩的 checkpoint 时，会自动进行解压缩，并从 checkpoint 恢复内存状态和体系结构状态。

In [ ]:
%%bash
source 07-checkpoint-env.sh >/dev/null

CHECKPOINT=$(find ${RESULT_PATH}/checkpoint/${WORKLOAD} -type f -name "*_.gz" | tail -1)

${READY2RUN_HOME}/emu \
    -i ${CHECKPOINT} \
    --diff ${NOOP_HOME}/ready-to-run/riscv64-nemu-interpreter-so \
    --max-cycles=50000 \
    2>/dev/null


In [ ]:
import os
import re
import json
from pathlib import Path
from itertools import product

app_list = [
    "bwaves", "gamess_cytosine", "gamess_gradient", "gamess_triazolium",
    "milc", "zeusmp", "gromacs", "cactusADM", "leslie3d", "namd", "dealII",
    "soplex_pds-50", "soplex_ref", "povray", "calculix", "GemsFDTD", "tonto",
    "lbm", "wrf", "sphinx3"
]

spec_2017_list = [
    "bwaves_1", "bwaves_2", "bwaves_3", "bwaves_4", "cactuBSSN", "namd",
    "parest", "povray", "lbm", "wrf", "blender", "cam4", "imagick", "nab",
    "fotonik3d", "roms", "perlbench_diff", "perlbench_spam", "perlbench_split",
    "gcc_pp_O2", "gcc_pp_O3", "gcc_ref32_O3", "gcc_ref32_O5", "gcc_small_O3",
    "mcf", "omnetpp", "xalancbmk", "x264_pass1", "x264_pass2", "x264_seek",
    "deepsjeng", "leela", "exchange2", "xz_cld", "xz_combined", "xz_cpu2006"
]

spec2017_int_list = [
    "perlbench_diff", "perlbench_spam", "perlbench_split", "gcc_pp_O2",
    "gcc_pp_O3", "gcc_ref32_O3", "gcc_ref32_O5", "gcc_small_O3", "mcf",
    "omnetpp", "xalancbmk", "x264_pass1", "x264_pass2", "x264_seek",
    "deepsjeng", "leela", "exchange2", "xz_cld", "xz_combined", "xz_cpu2006"
]

spec2017_fp_list = list(set(spec_2017_list) - set(spec2017_int_list))


def profiling_instrs(profiling_log, spec_app, using_new_script=False):
    regex = r".*total guest instructions = (.*)\x1b.*"
    new_path = os.path.join(profiling_log, spec_app, "profiling.out.log")
    old_path = os.path.join(profiling_log, "{}-out.txt".format(spec_app))

    if using_new_script:
        path = new_path
    else:
        path = old_path

    with open(path, "r", encoding="utf-8") as f:
        for i in f.readlines():
            if "total guest instructions" in i:
                match = re.findall(regex, i)
                match = match[0].replace(',', '')
                return match
        return 0


def cluster_weight(cluster_path, spec_app):
    points = {}
    weights = {}

    weights_path = f"{cluster_path}/{spec_app}/weights0"
    simpoints_path = f"{cluster_path}/{spec_app}/simpoints0"

    with open(weights_path, "r") as f:
        for line in f.readlines():
            a, b = line.split()
            weights.update({"{}".format(b): "{}".format(a)})

    with open(simpoints_path, "r") as f:
        for line in f.readlines():
            a, b = line.split()
            points.update({a: weights.get(b)})

    return points


def per_checkpoint_generate_json(profiling_log, cluster_path, app_list,
                                 target_path):
    result = {}
    for spec in app_list:
        result.update({
            spec: {
                "insts": profiling_instrs(profiling_log, spec),
                'points': cluster_weight(cluster_path, spec)
            }
        })
    with open(os.path.join(target_path), "w") as f:
        f.write(json.dumps(result))


def per_checkpoint_generate_worklist(cpt_path, target_path):
    cpt_path = cpt_path + "/"
    checkpoints = []
    for item in os.scandir(cpt_path):
        if item.is_dir():
            checkpoints.append(item.path)

    checkpoint_dirs = []
    for item in checkpoints:
        for entry in os.scandir(item):
            checkpoint_dirs.append(entry.path)

    with open(target_path, "w") as f:
        for i in checkpoint_dirs:
            path = i.replace(cpt_path, "")
            name = path.replace('/', "_", 1)
            print("{} {} 0 0 20 20".format(name, path), file=f)


def generate_result_list(base_path, times, ids):
    result_list = []

    for i, j, k in product(range(ids[0], times[0]), range(ids[1], times[1]),
                           range(ids[2], times[2])):
        cluster = f"cluster"
        profiling = f"profiling"
        checkpoint = f"checkpoint"
        result_list.append({
            "cl_res": os.path.join(base_path, "result", cluster),
            "profiling_log": os.path.join(base_path, "logs", profiling),
            "checkpoint_path": os.path.join(base_path, "result", checkpoint),
            "json_path": os.path.join(base_path, "result", checkpoint, f"{cluster}.json"),
            "list_path": os.path.join(base_path, "result", checkpoint, "checkpoint.lst"),
        })

    print("Result list:")
    print(json.dumps(result_list, indent=2, separators=(",", ": ")))
    return result_list



def dump_result(base_path, spec_app_list, times, ids):
    result_list = generate_result_list(base_path, times, ids)

    for result in result_list:
        per_checkpoint_generate_json(result["profiling_log"], result["cl_res"],
                                     spec_app_list, result["json_path"])
        per_checkpoint_generate_worklist(result["checkpoint_path"],
                                         result["list_path"])


# NOTE: should be same with 07-checkpoint-env.sh
spec_list=["stream_100000"]
base_path = os.path.join(os.getcwd(), "07-checkpoint")
times = [1, 1, 1]
ids = [0, 0, 0]

dump_result(base_path, spec_list, times, ids)

# 计数器

香山实现了全面的性能计数器，分为以下三种：

- Accumulation：基础的累加型计数器（log 打印）
- Histogram：统计数值分布（log 打印）
- Rolling：采集片段计数以分析性能变化（ChiselDB 存储）

Accumulation 类型的性能计数器在仿真结束后打印到 stderr。在此之前的每次执行都会输出这些数据，只不过我们都将他们丢弃了。现在，我们来看一看这些数据。由于这些数据太大了，因此我们只展示最后的 100 条

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null
cd ${NOOP_HOME}

${READY2RUN_HOME}/emu \
    -i ${READY2RUN_HOME}/hello-riscv64-xs.bin \
    --no-diff 2>stderr.log

echo "=== Last 10 lines:"
tail -n 10 stderr.log

echo "=== Example of XSPerfAccumulate: rob commitInstr"
grep -n "rob: commitInstr," stderr.log

echo "=== Example of XSHistogram: l2cache acquire period"
grep -n "l2cache.slices_0.mshrCtl: acquire_period" stderr.log

在 [https://github.com/OpenXiangShan/env-scripts/blob/main/perf/perf.py](https://github.com/OpenXiangShan/env-scripts/blob/main/perf/perf.py) 可以找到更多的数据分析脚本

Rolling 采用滑动窗口的思想，采集程序运行过程中每个片段（i.e. 每 10000 周期）的性能数据，用于分析程序行为特征和变化。

要启用 RollingDB，需要在编译时指定 `WITH_ROLLINGDB=1`，并在运行时指定 `--dump-db` 参数。本次 tutorial 中受时间和运算资源限制，请不要运行编译和运行的代码。

我们提供了用此方式采集得到的 `xs-perf-rolling.db` 文件，您可以使用 `${NOOP_HOME}/scripts/rolling` 下的 python 脚本对其进行分析。您可以尝试运行下面的单元格。

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null

# 编译和运行：注意，请不要运行这部分
# cd ${NOOP_HOME}
# make clean
# make emu EMU_THREADS=4 WITH_CHISELDB=1 WITH_ROLLINGDB=1 -j8 \
#     PGO_WORKLOAD=${NOOP_HOME}/ready-to-run/coremark-2-iteration.bin \
#     PGO_MAX_CYCLE=10000 PGO_EMU_ARGS=--no-diff LLVM_PROFDATA=llvm-profdata
# ./build/emu -i ./ready-to-run/coremark-2-iteration.bin \
#     --diff ./ready-to-run/riscv64-nemu-interpreter-so --dump-db
# cp `find $NOOP_HOME/build/ -type f -name "*.db" | tail -1` \
#     ${XS_PROJECT_ROOT}/tutorial/${dir}/xs-perf-rolling.db

# 使用脚本对 rolling db 进行分析
cd ${NOOP_HOME}/scripts/rolling
python3 rollingplot.py ${READY2RUN_HOME}/xs-perf-rolling.db ipc

该脚本会输出以下图片，可以看到 XiangShan 在运行这一程序时，每一段时间内的 IPC 变化：

![perf](../XiangShan//scripts/rolling/results/perf.png)

# Topdown

香山在 RTL 上实现了完善的 Topdown 计数器，并针对 RISC-V 和香山微架构进行了优化

Topdown 计数器的结果在仿真结束后会输出到 stderr。您可以使用我们准备好的脚本对香山的 Topdown 结果进行分析

由于 SPEC2006 的切片数量非常多，且性能相关的数据量很大，以下脚本会涉及大量文件读写，因此我们不在现场执行这些分析脚本

In [ ]:
%%bash
cd ../ && source env.sh >/dev/null

# 请不要执行以下命令！
# cd ${NOOP_HOME}/scripts/top-down && python3 top_down.py -s ${READY2RUN_HOME}/SPEC06_EmuTasks_topdown -j ${READY2RUN_HOME}/SPEC06_EmuTasks_topdown.json
# ls ${NOOP_HOME}/scripts/top-down/results